# SmartVision Lite — Custom YOLO Training

This Google Colab notebook trains a lightweight Ultralytics YOLO detector on a dataset in YOLO format. Run the cells in order. Training speed and quality depend on the dataset and selected Colab hardware; no accuracy value is guaranteed.

## 1. Install Ultralytics
Select **Runtime → Change runtime type → T4 GPU** when a GPU is available, then install the packages.

In [ ]:
%pip install -q "ultralytics>=8.3,<9" pyyaml

In [ ]:
from pathlib import Path
import shutil
import zipfile
import yaml
import torch
from IPython.display import Image, display
from PIL import Image as PILImage
from google.colab import files
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Provide a dataset

Export a dataset in **YOLO** format. A Roboflow-exported ZIP works, but download it yourself from Roboflow and upload it here—do not paste or store an API key in this notebook. You may instead mount Google Drive and point `DATASET_YAML` at an existing file.

Expected folders are `train/images`, `train/labels`, `valid/images`, and `valid/labels`, plus a YAML file. Set `UPLOAD_ZIP = True` to upload and extract one ZIP.

In [ ]:
UPLOAD_ZIP = False
EXTRACT_TO = Path("/content/dataset")

if UPLOAD_ZIP:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one dataset ZIP file.")
    EXTRACT_TO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_names[0]) as archive:
        archive.extractall(EXTRACT_TO)
    print(f"Extracted dataset to {EXTRACT_TO}")

# Change this path if your export uses data.yaml or is stored elsewhere.
DATASET_YAML = Path("/content/dataset/dataset.yaml")
print(f"Dataset YAML: {DATASET_YAML}")

## 3. Verify `dataset.yaml`
This catches common path, split, and class-name mistakes before a long training run.

In [ ]:
if not DATASET_YAML.is_file():
    candidates = list(EXTRACT_TO.rglob("*.yaml")) + list(EXTRACT_TO.rglob("*.yml"))
    raise FileNotFoundError(
        f"Could not find {DATASET_YAML}. Available YAML files: {candidates}"
    )

with DATASET_YAML.open(encoding="utf-8") as handle:
    dataset_config = yaml.safe_load(handle)

required = {"train", "val", "names"}
missing = required.difference(dataset_config or {})
if missing:
    raise ValueError(f"Dataset YAML is missing keys: {sorted(missing)}")
if not isinstance(dataset_config["names"], (list, dict)) or not dataset_config["names"]:
    raise ValueError("'names' must define at least one class.")

print(yaml.safe_dump(dataset_config, sort_keys=False))
print(f"Classes: {len(dataset_config['names'])}")
print("Dataset configuration looks structurally valid.")

## 4. Train a lightweight model
Increase epochs only after the pipeline works. Colab may download the nano pretrained checkpoint on the first run.

In [ ]:
model = YOLO("yolo11n.pt")
train_results = model.train(
    data=str(DATASET_YAML),
    epochs=30,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/content/runs/detect",
    name="smartvision-lite",
    patience=10,
    plots=True,
)

## 5. Validate and inspect precision, recall, and mAP

Precision measures how many predicted objects are correct. Recall measures how many labeled objects were found. mAP50 uses an IoU threshold of 0.50; mAP50–95 averages stricter thresholds from 0.50 through 0.95. Compare models on the same held-out split.

In [ ]:
validation = model.val(data=str(DATASET_YAML), plots=True)
print(f"Precision: {validation.box.mp:.4f}")
print(f"Recall: {validation.box.mr:.4f}")
print(f"mAP50: {validation.box.map50:.4f}")
print(f"mAP50-95: {validation.box.map:.4f}")

## 6. Display the confusion matrix
Rows and columns compare actual and predicted classes. Inspect both the normalized and raw plots in the validation output directory.

In [ ]:
validation_dir = Path(validation.save_dir)
for filename in ("confusion_matrix_normalized.png", "confusion_matrix.png"):
    plot_path = validation_dir / filename
    if plot_path.is_file():
        display(Image(filename=str(plot_path)))
    else:
        print(f"Plot not generated: {plot_path}")

## 7. Export `best.pt`
Download the best checkpoint to your computer. Model weights should not be committed to Git.

In [ ]:
best_path = Path(model.trainer.best)
if not best_path.is_file():
    raise FileNotFoundError(f"Training completed but best.pt was not found at {best_path}")
print(f"Best checkpoint: {best_path}")
files.download(str(best_path))

## 8. Example inference
Set `EXAMPLE_IMAGE` to a test image not used for training, then run inference with the exported best checkpoint.

In [ ]:
EXAMPLE_IMAGE = Path("/content/dataset/test/images/example.jpg")
if not EXAMPLE_IMAGE.is_file():
    print(f"Update EXAMPLE_IMAGE before inference; file not found: {EXAMPLE_IMAGE}")
else:
    best_model = YOLO(str(best_path))
    predictions = best_model.predict(
        source=str(EXAMPLE_IMAGE), conf=0.25, save=True, project="/content/inference"
    )
    annotated = predictions[0].plot()[:, :, ::-1]
    display(PILImage.fromarray(annotated))
    print("Inference output was also saved under /content/inference")

### Next step
Copy the downloaded `best.pt` into the local SmartVision Lite `models/` directory, choose **Custom model** in the sidebar, and test it on data that was not used for training.